## __Tentative de fine tuning de mobilenetV2__

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchmetrics.classification import MulticlassF1Score
from lib.data.preprocessing import TorchPreprocessor
from lib.data.dataset import BeeDataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch.optim as optim

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"L'entraînement se fera sur : {device}")

2026-03-02 11:38:40.497864: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


L'entraînement se fera sur : cuda


In [2]:
# Load model directly from Hugging Face
num_labels = 50
base_processor = AutoImageProcessor.from_pretrained("google/mobilenet_v2_1.0_224")
model = AutoModelForImageClassification.from_pretrained("google/mobilenet_v2_1.0_224", num_labels=num_labels,ignore_mismatched_sizes=True)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([50]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([50, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Le preprocessor fourni ne pad pas, ce qui nous fait perdre des abeilles. On se propose donc de remplacer cette partie du code

In [3]:
mobile_net_processor_parameters= {
    "mean" : base_processor.image_mean,
    "std" : base_processor.image_std,
    "crop_size" : (base_processor.crop_size["height"], base_processor.crop_size["width"])
}

preprocessor = TorchPreprocessor(
    normalize=True,
    resize_method="pad",
    target_size=mobile_net_processor_parameters["crop_size"],
)

dataset = BeeDataset(
    root_dir="data/train",
    transform=preprocessor
)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [7]:
# 1. Configuration (Backbone gelé, Tête active)
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

# Filtrer les paramètres pour l'optimiseur (plus propre pour la mémoire)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

# 2. Boucle d'entraînement
model.train()
num_epochs = 10
dataloader = DataLoader(dataset, batch_size=100, shuffle=True)

f1_metric = MulticlassF1Score(num_classes=num_labels, average='macro').to(device)
model.to(device)

for epoch in range(num_epochs):
    loop = tqdm(dataloader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    
    epoch_loss = 0
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        # --- Calcul du F1 Score ---
        preds = torch.argmax(logits, dim=-1)
        f1_metric.update(preds, labels)
        
        # Affichage dynamique
        current_f1 = f1_metric.compute()
        epoch_loss += loss.item()
        loop.set_postfix(
            loss=f"{loss.item():.4f}", 
            f1=f"{current_f1.item():.4f}"
        )
    # Score final de l'epoch
    final_f1 = f1_metric.compute()
    print(f"-> Fin Epoch {epoch+1} | Loss: {epoch_loss/len(dataloader):.4f} | F1: {final_f1:.4f}")

Epoch [1/10]:   0%|          | 0/80 [00:00<?, ?it/s]

-> Fin Epoch 1 | Loss: 1.8063 | F1: 0.2150


Epoch [2/10]:   0%|          | 0/80 [00:00<?, ?it/s]

-> Fin Epoch 2 | Loss: 1.5178 | F1: 0.3447


Epoch [3/10]:   0%|          | 0/80 [00:00<?, ?it/s]

...

-> Fin Epoch 8 | Loss: 1.0846 | F1: 0.6782


Epoch [9/10]:   0%|          | 0/80 [00:00<?, ?it/s]

-> Fin Epoch 9 | Loss: 1.0666 | F1: 0.6975


Epoch [10/10]:   0%|          | 0/80 [00:00<?, ?it/s]

-> Fin Epoch 10 | Loss: 1.0344 | F1: 0.7129


In [9]:
class JITWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        # On force le retour d'un Tensor pur (les logits)
        outputs = self.model(x)
        return outputs.logits

# On trace le wrapper
wrapper = JITWrapper(model).eval()
traced_model = torch.jit.trace(wrapper, dummy_input)
traced_model.save("mobilenet_v2_bees_jit.pt")